# Week 3 Benchmark Report

This notebook exercises the synthetic Track 2 benchmark for v9: ring-level metrics, bootstrap confidence intervals, a topology-matched null baseline, and ablation summaries.

The synthetic benchmark is deliberately small and self-contained so it can run in the repo without external data.

In [1]:
from __future__ import annotations

import sys
import time
import tracemalloc
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = next((candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / 'src').exists()), Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import aggregate_scenario_results, bootstrap_ci, performance_profile_row, ring_metrics, sample_topology_matched_null_rings
from financial_graph_engine import FinancialGraphEngine, is_decaying_ring

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [3]:
def make_week3_scenarios() -> list[dict[str, object]]:
    scenarios: list[dict[str, object]] = []
    base_dates = pd.date_range('2024-02-01', periods=24, freq='2h')

    for scenario_index, seed in enumerate([11, 22, 33], start=1):
        rng = np.random.default_rng(seed)
        prefix = f'S{scenario_index}'
        a, b, c, d = [f'{prefix}_{suffix}' for suffix in ('A', 'B', 'C', 'D')]
        e, f, g = [f'{prefix}_{suffix}' for suffix in ('E', 'F', 'G')]

        transactions = [
            {'sender_id': a, 'receiver_id': b, 'amount': 120.0, 'timestamp': base_dates[0]},
            {'sender_id': b, 'receiver_id': c, 'amount': 110.0, 'timestamp': base_dates[1]},
            {'sender_id': c, 'receiver_id': a, 'amount': 100.0, 'timestamp': base_dates[2]},
            {'sender_id': a, 'receiver_id': b, 'amount': 118.0, 'timestamp': base_dates[3]},
            {'sender_id': b, 'receiver_id': d, 'amount': 108.0, 'timestamp': base_dates[4]},
            {'sender_id': d, 'receiver_id': a, 'amount': 98.0, 'timestamp': base_dates[5]},
            {'sender_id': e, 'receiver_id': f, 'amount': 10.0 + scenario_index, 'timestamp': base_dates[6]},
            {'sender_id': f, 'receiver_id': g, 'amount': 20.0 + scenario_index, 'timestamp': base_dates[7]},
            {'sender_id': g, 'receiver_id': e, 'amount': 30.0 + scenario_index, 'timestamp': base_dates[8]},
        ]

        for fanout_index in range(6):
            transactions.append(
                {
                    'sender_id': b,
                    'receiver_id': f'{prefix}_NOISE_{fanout_index}',
                    'amount': 40.0 + fanout_index + rng.normal(0, 1),
                    'timestamp': base_dates[9 + fanout_index],
                }
            )

        scenarios.append(
            {
                'scenario_name': f'scenario_{scenario_index}',
                'transactions': pd.DataFrame(transactions),
                'truth_rings': [[a, b, c, d]],
            }
        )

    return scenarios


scenarios = make_week3_scenarios()
[scenario['scenario_name'] for scenario in scenarios]

['scenario_1', 'scenario_2', 'scenario_3']

In [4]:
def node_list_from_temporal_rings(temporal_rings: list[list[tuple[str, str, object, float]]]) -> list[list[str]]:
    node_lists: list[list[str]] = []
    for ring in temporal_rings:
        nodes = sorted({hop[0] for hop in ring} | {hop[1] for hop in ring})
        node_lists.append(nodes)
    return node_lists


def run_pipeline(
    transactions: pd.DataFrame,
    scenario_name: str,
    degree_ratio_threshold: float,
    apply_decay: bool,
    dedupe: bool,
) -> tuple[list[list[str]], dict[str, object]]:
    engine = FinancialGraphEngine(degree_ratio_threshold=degree_ratio_threshold)
    tracemalloc.start()
    start = time.perf_counter()
    engine.build_graph_from_transactions(transactions)
    raw_temporal_rings = engine.detect_temporal_rings(max_cycle_length=4)
    if apply_decay:
        raw_temporal_rings = [ring for ring in raw_temporal_rings if is_decaying_ring(ring)]
    if dedupe:
        detected_rings = engine._dedupe_into_rings(raw_temporal_rings)
    else:
        detected_rings = node_list_from_temporal_rings(raw_temporal_rings)
    runtime_seconds = time.perf_counter() - start
    _, peak_memory_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    profile_row = performance_profile_row(engine, scenario_name, runtime_seconds, peak_memory_bytes=peak_memory_bytes)
    return detected_rings, profile_row

In [5]:
configs = [
    {'config_name': 'baseline', 'degree_ratio_threshold': 0.3, 'apply_decay': True, 'dedupe': True},
    {'config_name': 'no_degree_prune', 'degree_ratio_threshold': 1.0, 'apply_decay': True, 'dedupe': True},
    {'config_name': 'no_dedup', 'degree_ratio_threshold': 1.0, 'apply_decay': True, 'dedupe': False},
    {'config_name': 'no_decay', 'degree_ratio_threshold': 1.0, 'apply_decay': False, 'dedupe': True},
]

scenario_pairs_by_config: dict[str, list[tuple[list[list[str]], list[list[str]]]]] = {config['config_name']: [] for config in configs}
scenario_rows: list[dict[str, object]] = []
performance_rows: list[dict[str, object]] = []
null_rows: list[dict[str, object]] = []

for scenario in scenarios:
    truth_rings = scenario['truth_rings']
    transactions = scenario['transactions']
    for config in configs:
        detected_rings, profile_row = run_pipeline(
            transactions=transactions,
            scenario_name=f"{scenario['scenario_name']}::{config['config_name']}",
            degree_ratio_threshold=config['degree_ratio_threshold'],
            apply_decay=config['apply_decay'],
            dedupe=config['dedupe'],
        )
        metrics = ring_metrics(detected_rings, truth_rings)
        row = {
            'scenario': scenario['scenario_name'],
            'config': config['config_name'],
            **metrics,
        }
        scenario_rows.append(row)
        performance_rows.append({**profile_row, 'config': config['config_name']})
        scenario_pairs_by_config[config['config_name']].append((detected_rings, truth_rings))

    baseline_engine = FinancialGraphEngine(degree_ratio_threshold=0.3)
    baseline_engine.build_graph_from_transactions(transactions)
    null_rings = sample_topology_matched_null_rings(baseline_engine.graph, truth_rings, seed=101)
    null_metrics = ring_metrics(null_rings, truth_rings)
    null_rows.append({'scenario': scenario['scenario_name'], **null_metrics})

scenario_results = pd.DataFrame(scenario_rows)
performance_table = pd.DataFrame(performance_rows)
null_baseline_table = pd.DataFrame(null_rows)
scenario_results

,scenario,config,precision,recall,f1,account_recall,detected_rings,truth_rings
0,scenario_1,baseline,0.0,0.0,0.000000,0.0,1.0,1.0
1,scenario_1,no_degree_prune,0.0,0.0,0.000000,0.0,1.0,1.0
2,scenario_1,no_dedup,0.0,0.0,0.000000,0.0,1.0,1.0
3,scenario_1,no_decay,0.5,1.0,0.666667,1.0,2.0,1.0
4,scenario_2,baseline,1.0,1.0,1.000000,1.0,1.0,1.0
5,scenario_2,no_degree_prune,1.0,1.0,1.000000,1.0,1.0,1.0
6,scenario_2,no_dedup,0.0,0.0,0.000000,0.0,2.0,1.0
7,scenario_2,no_decay,0.5,1.0,0.666667,1.0,2.0,1.0
8,scenario_3,baseline,1.0,1.0,1.000000,1.0,1.0,1.0
9,scenario_3,no_degree_prune,1.0,1.0,1.000000,1.0,1.0,1.0


In [6]:
def summarize_config(config_name: str, scenario_pairs: list[tuple[list[list[str]], list[list[str]]]]) -> dict[str, object]:
    aggregate = aggregate_scenario_results(scenario_pairs)

    def metric_fn(metric_name: str):
        return lambda samples: aggregate_scenario_results(samples)[metric_name]

    precision_point, precision_lo, precision_hi = bootstrap_ci(scenario_pairs, metric_fn('precision'), n_resamples=500, seed=7)
    recall_point, recall_lo, recall_hi = bootstrap_ci(scenario_pairs, metric_fn('recall'), n_resamples=500, seed=7)
    f1_point, f1_lo, f1_hi = bootstrap_ci(scenario_pairs, metric_fn('f1'), n_resamples=500, seed=7)
    account_point, account_lo, account_hi = bootstrap_ci(scenario_pairs, metric_fn('account_recall'), n_resamples=500, seed=7)

    return {
        'config': config_name,
        'precision': aggregate['precision'],
        'precision_ci': f"[{precision_lo:.3f}, {precision_hi:.3f}]",
        'recall': aggregate['recall'],
        'recall_ci': f"[{recall_lo:.3f}, {recall_hi:.3f}]",
        'f1': aggregate['f1'],
        'f1_ci': f"[{f1_lo:.3f}, {f1_hi:.3f}]",
        'account_recall': aggregate['account_recall'],
        'account_recall_ci': f"[{account_lo:.3f}, {account_hi:.3f}]",
        'detected_rings': int(aggregate['detected_rings']),
        'truth_rings': int(aggregate['truth_rings']),
    }

summary_rows = [summarize_config(config_name, pairs) for config_name, pairs in scenario_pairs_by_config.items()]
summary_table = pd.DataFrame(summary_rows)
summary_table

,config,precision,precision_ci,recall,recall_ci,f1,f1_ci,account_recall,account_recall_ci,detected_rings,truth_rings
0,baseline,0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",3,3
1,no_degree_prune,0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",3,3
2,no_dedup,0.000000,"[0.000, 0.000]",0.000000,"[0.000, 0.000]",0.000000,"[0.000, 0.000]",0.000000,"[0.000, 0.000]",5,3
3,no_decay,0.500000,"[0.500, 0.500]",1.000000,"[1.000, 1.000]",0.666667,"[0.667, 0.667]",1.000000,"[1.000, 1.000]",6,3


In [7]:
print('Scenario-level metrics by config:')
display(scenario_results)

print('Bootstrap summary by config:')
display(summary_table)

print('Topology-matched null baseline:')
display(null_baseline_table)

print('Performance profile:')
display(performance_table)

Scenario-level metrics by config:


,scenario,config,precision,recall,f1,account_recall,detected_rings,truth_rings
0,scenario_1,baseline,0.0,0.0,0.000000,0.0,1.0,1.0
1,scenario_1,no_degree_prune,0.0,0.0,0.000000,0.0,1.0,1.0
2,scenario_1,no_dedup,0.0,0.0,0.000000,0.0,1.0,1.0
3,scenario_1,no_decay,0.5,1.0,0.666667,1.0,2.0,1.0
4,scenario_2,baseline,1.0,1.0,1.000000,1.0,1.0,1.0
5,scenario_2,no_degree_prune,1.0,1.0,1.000000,1.0,1.0,1.0
6,scenario_2,no_dedup,0.0,0.0,0.000000,0.0,2.0,1.0
7,scenario_2,no_decay,0.5,1.0,0.666667,1.0,2.0,1.0
8,scenario_3,baseline,1.0,1.0,1.000000,1.0,1.0,1.0
9,scenario_3,no_degree_prune,1.0,1.0,1.000000,1.0,1.0,1.0


Bootstrap summary by config:


,config,precision,precision_ci,recall,recall_ci,f1,f1_ci,account_recall,account_recall_ci,detected_rings,truth_rings
0,baseline,0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",3,3
1,no_degree_prune,0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",0.666667,"[0.000, 1.000]",3,3
2,no_dedup,0.000000,"[0.000, 0.000]",0.000000,"[0.000, 0.000]",0.000000,"[0.000, 0.000]",0.000000,"[0.000, 0.000]",5,3
3,no_decay,0.500000,"[0.500, 0.500]",1.000000,"[1.000, 1.000]",0.666667,"[0.667, 0.667]",1.000000,"[1.000, 1.000]",6,3


Topology-matched null baseline:


,scenario,precision,recall,f1,account_recall,detected_rings,truth_rings
0,scenario_1,0.0,0.0,0.0,0.0,1.0,1.0
1,scenario_2,0.0,0.0,0.0,0.0,1.0,1.0
2,scenario_3,0.0,0.0,0.0,0.0,1.0,1.0


Performance profile:


,scenario,scenario_edges,scenario_nodes,nodes_after_scc_filter,nodes_after_degree_prune,candidates_generated_at_least,candidates_processed,accepted_temporal_rings,truncated,runtime_seconds,peak_memory_mb,config
0,scenario_1::baseline,15,13,7,7,3,3,3,False,0.768845,0.145706,baseline
1,scenario_1::no_degree_prune,15,13,7,7,3,3,3,False,0.322444,0.037793,no_degree_prune
2,scenario_1::no_dedup,15,13,7,7,3,3,3,False,0.272063,0.035333,no_dedup
3,scenario_1::no_decay,15,13,7,7,3,3,3,False,0.377030,0.035840,no_decay
4,scenario_2::baseline,15,13,7,7,3,3,3,False,0.455122,0.037610,baseline
5,scenario_2::no_degree_prune,15,13,7,7,3,3,3,False,0.407429,0.035111,no_degree_prune
6,scenario_2::no_dedup,15,13,7,7,3,3,3,False,0.379935,0.034841,no_dedup
7,scenario_2::no_decay,15,13,7,7,3,3,3,False,0.392800,0.035142,no_decay
8,scenario_3::baseline,15,13,7,7,3,3,3,False,0.231416,0.036637,baseline
9,scenario_3::no_degree_prune,15,13,7,7,3,3,3,False,0.100182,0.034493,no_degree_prune


## Notes

- The baseline uses SCC-first pruning, degree-ratio pruning, temporal validation, decay filtering, and deduplication.
- The no-dedup and no-decay rows are computed on the same synthetic scenarios to make their effect visible in a small self-contained benchmark.
- The null baseline is topology-matched only in the limited synthetic sense used here: same ring sizes and degree-weighted node sampling from the scenario graph.
- In the final write-up, replace this synthetic table with the real held-out AMLSim scenarios and preserve the same report structure.